# 999: Combine the Team Datasets into One Table

This notebook merges the outputs of notebooks 01 and 02 into **one team-performance table**, with one row per team + season. That table gets joined onto the player tables (stats, salary, ...) on `season` + `team` before XGBoost, so every player row gets their team's results for that season.

| Input (`data-files/02-processed/`) | From | Row = |
|---|---|---|
| `llimllib_team_playoff_features.csv` | 01 | team + season: wins, playoff results, 5-year playoff trends |
| `espn_standings_features.csv` | 02 | team + season: official regular-season standing and final seed |

**Output:** `data-files/02-processed/team_performance_combined.csv`

*Run order: `01` → `02` → `999`.*

## Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path('..') / 'salary-data'))
from nba_cleaning_utils import PROCESSED, FIRST_SEASON, LAST_SEASON

playoffs = pd.read_csv(PROCESSED / 'llimllib_team_playoff_features.csv', dtype={'team_id': str})
standings = pd.read_csv(PROCESSED / 'espn_standings_features.csv', dtype={'espn_team_id': str})

print('llimllib team rows:', playoffs.shape)
print('ESPN standings rows:', standings.shape)

llimllib team rows: (330, 11)
ESPN standings rows: (330, 8)


## 1. Choose the base table

The llimllib table from 01 is the base. It has the NBA `team_id` and full team name, and most of the features. ESPN's standing columns get **left-joined** onto it. Both tables should cover the same 330 team-seasons, so the join should match every row.

## 2. Join keys

Both tables use `season` (end year) + `team` (NBA abbreviation; notebook 02 already mapped ESPN's abbreviations). Before joining, check that the key is unique on both sides and that both sides contain the same team-seasons.

In [2]:
key = ['season', 'team']
print('duplicate keys, llimllib:', playoffs.duplicated(key).sum(), '| ESPN:', standings.duplicated(key).sum())
pairs = playoffs[key].merge(standings[key], how='outer', indicator=True)['_merge'].value_counts()
pairs

duplicate keys, llimllib: 0 | ESPN: 0


_merge
both          330
left_only       0
right_only      0
Name: count, dtype: int64

## 3. Merge

In [3]:
merged = playoffs.merge(standings.rename(columns={'conference': 'espn_conference'}),
                        on=key, how='left', validate='one_to_one', indicator=True)
print('rows:', len(merged), '| matched:', (merged['_merge'] == 'both').sum())
merged = merged.drop(columns='_merge')

rows: 330 | matched: 330


## 4. Check the merge

**(a) Do the two sources agree on each team's record and conference?** 01's record comes from Basketball-Reference and 02's from ESPN. If both match, the join attached the right teams.

In [4]:
print('conference differs:', (merged['conference'] != merged['espn_conference']).sum())
print('wins differ:', (merged['wins'] != merged['espn_wins']).sum())
espn_pct = merged['espn_wins'] / (merged['espn_wins'] + merged['espn_losses'])
print('win % differs by more than 0.001:', ((merged['win_pct'] - espn_pct).abs() > 1e-3).sum())

conference differs: 0
wins differ: 0
win % differs by more than 0.001: 0


**(b) Do the seeds agree with playoff qualification?** Seeds 1-8 after the play-in are exactly the 16 playoff teams. In the regular-season standing, the top 6 always make the playoffs and 11th place or lower never does. 7th-10th depends on the play-in, from 2020-21 on.

In [5]:
print('conference_seed <= 8 disagrees with made_playoffs:', ((merged['conference_seed'] <= 8) != merged['made_playoffs']).sum())
print('regular_season_standing 1-6 that missed the playoffs:', (merged['regular_season_standing'].le(6) & ~merged['made_playoffs']).sum())
print('regular_season_standing 11+ that made the playoffs:', (merged['regular_season_standing'].ge(11) & merged['made_playoffs']).sum())

pd.crosstab(merged['regular_season_standing'], merged['made_playoffs']).loc[6:11].T

conference_seed <= 8 disagrees with made_playoffs: 0
regular_season_standing 1-6 that missed the playoffs: 0
regular_season_standing 11+ that made the playoffs: 0


regular_season_standing,6,7,8,9,10,11
made_playoffs,,,,,,
False,0,0,4,19,21,22
True,22,22,18,3,1,0


**What we found:** Both sources agree on every team's record and conference, and the seeds line up with `made_playoffs`. Every team seeded 1-8 after the play-in made the playoffs, and every team seeded 9-15 didn't. In the regular-season standing, 7th-10th is where the two diverge. Every 7th-place team made it, but **4 of the 22 8th-place teams lost their way out** in the play-in, and **4 teams that finished 9th or 10th played their way in** (3 ninth-place teams and 1 tenth-place team).

**(c) What do the official standings change compared with ranking by win %?** Before notebook 02, the standing was approximated by ranking win % within each conference, with ties sharing the same rank. We rebuild that approximation here only to see how much the official standings correct it.

In [6]:
approx_rank = (merged.groupby(['season', 'conference'])['win_pct']
                     .rank(method='min', ascending=False).astype(int))
tied = merged.duplicated(['season', 'conference', 'wins'], keep=False) & merged.duplicated(['season', 'conference', 'win_pct'], keep=False)
differs = approx_rank != merged['regular_season_standing']

print(f"tied records: {merged[tied].groupby(['season', 'conference', 'wins']).ngroups} groups covering {tied.sum()} team-seasons")
print('team-seasons where the win-% rank differs from the official standing:', differs.sum())
print('...all of them in a tied group:', (differs <= tied).all())

(merged.assign(approx_rank=approx_rank)
       .query("season == 2024 and conference == 'East' and 4 <= regular_season_standing <= 9")
       .sort_values('regular_season_standing')
       [['team', 'wins', 'approx_rank', 'regular_season_standing', 'conference_seed', 'made_playoffs']])

tied records: 42 groups covering 93 team-seasons
team-seasons where the win-% rank differs from the official standing: 51
...all of them in a tied group: True


,team,wins,approx_rank,regular_season_standing,conference_seed,made_playoffs
245,CLE,48,4,4,4,True
261,ORL,47,5,5,5,True
251,IND,47,5,6,6,True
262,PHI,47,5,7,7,True
255,MIA,46,8,8,8,True
244,CHI,39,9,9,9,False


**What we found:** There are **42 groups of identical records in the same conference, covering 93 team-seasons**. The win-% rank gave every team in a group the same rank, so it was wrong for 51 of them, and only teams with a tied record were affected. The official standing breaks every tie. In the 2023-24 East, ORL, IND, and PHI all went 47-35. The win-% rank put all three at 5. Officially they were 5th, 6th, and 7th, so PHI had to go through the play-in.

## 5. Final columns

Keep the keys and the features. Drop the columns that were only used for checking (`espn_conference`, `espn_wins`, `espn_losses`) and `espn_team_id`, which isn't needed for any join.

In [7]:
final_cols = [
    # keys
    'season', 'team', 'team_id', 'full_name', 'conference',
    # regular season
    'wins', 'win_pct', 'regular_season_standing', 'conference_seed',
    # playoffs
    'made_playoffs', 'furthest_playoff_round',
    # 5-year playoff trends (the five seasons before this one)
    'playoff_appearances_last_5', 'avg_playoff_round_last_5',
]
team_performance = merged[final_cols].sort_values(key).reset_index(drop=True)

tp = team_performance
checks = {
    'one row per season + team': not tp.duplicated(key).any(),
    f'330 rows: 30 teams in every season {FIRST_SEASON}-{LAST_SEASON}': len(tp) == 330 and tp.groupby('season')['team'].nunique().eq(30).all(),
    'no missing values': tp.notna().all().all(),
    'regular_season_standing is 1-15 in every conference-season': tp.groupby(['season', 'conference'])['regular_season_standing'].apply(lambda s: sorted(s) == list(range(1, 16))).all(),
    'conference_seed is 1-15 in every conference-season': tp.groupby(['season', 'conference'])['conference_seed'].apply(lambda s: sorted(s) == list(range(1, 16))).all(),
    'seeds 1-8 are exactly the playoff teams': ((tp['conference_seed'] <= 8) == tp['made_playoffs']).all(),
    'playoff round is 0-5, one champion per season': tp['furthest_playoff_round'].between(0, 5).all() and tp.groupby('season')['furthest_playoff_round'].apply(lambda r: (r == 5).sum()).eq(1).all(),
}
for name, ok in checks.items():
    print('PASS' if ok else 'FAIL', '-', name)
assert all(checks.values())

PASS - one row per season + team
PASS - 330 rows: 30 teams in every season 2016-2026
PASS - no missing values
PASS - regular_season_standing is 1-15 in every conference-season
PASS - conference_seed is 1-15 in every conference-season
PASS - seeds 1-8 are exactly the playoff teams
PASS - playoff round is 0-5, one champion per season


In [8]:
team_performance.drop(columns='season').describe().T[['count', 'mean', 'min', '50%', 'max']].round(2)

,count,mean,min,50%,max
wins,330.0,40.03,10.00,42.00,73.00
win_pct,330.0,0.50,0.12,0.51,0.89
regular_season_standing,330.0,8.00,1.00,8.00,15.00
conference_seed,330.0,8.00,1.00,8.00,15.00
furthest_playoff_round,330.0,1.03,0.00,1.00,5.00
playoff_appearances_last_5,330.0,2.67,0.00,3.00,5.00
avg_playoff_round_last_5,330.0,1.03,0.00,0.80,4.60


There are no missing values to handle, and every team-season has every feature.

## 6. Save

In [9]:
out = PROCESSED / 'team_performance_combined.csv'
team_performance.to_csv(out, index=False)
print('saved', team_performance.shape, 'to', out)
team_performance.head()

saved (330, 13) to ..\..\data-files\02-processed\team_performance_combined.csv


,season,team,team_id,full_name,conference,wins,win_pct,regular_season_standing,conference_seed,made_playoffs,furthest_playoff_round,playoff_appearances_last_5,avg_playoff_round_last_5
0,2016,ATL,1610612737,Atlanta Hawks,East,48,0.585366,4,4,True,2,5,1.6
1,2016,BKN,1610612751,Brooklyn Nets,East,21,0.256098,14,14,False,0,3,0.8
2,2016,BOS,1610612738,Boston Celtics,East,48,0.585366,5,5,True,1,4,1.4
3,2016,CHA,1610612766,Charlotte Hornets,East,48,0.585366,6,6,True,1,1,0.2
4,2016,CHI,1610612741,Chicago Bulls,East,42,0.512195,9,9,False,0,5,1.8


**Output:** `data-files/02-processed/team_performance_combined.csv`, **one row per team + season** (330 rows, 2015-16 to 2025-26).

| Feature | Column(s) |
|---|---|
| Regular-season final standing (seed) | `regular_season_standing` (official order before the play-in); `conference_seed` (seed after the play-in) |
| Games won | `wins`, `win_pct` |
| Made the playoffs | `made_playoffs` |
| Furthest playoff round | `furthest_playoff_round` (0 = missed, 1-3 = lost in that round, 4 = lost in the Finals, 5 = champion) |
| 5-year trends | `playoff_appearances_last_5`, `avg_playoff_round_last_5` |

**How to join it with the other datasets:** on `season` (end year, 2026 = 2025-26) + `team` (NBA abbreviation), as a **left join from the player table** (e.g. `salary_features_combined.csv`). Each player row gets their team's values for that season. Because we only predict offseason trades, the current season's standings and playoff results are already final when the trade window opens.